# TrialMatch browser app · Colab GPU launch

This notebook is an operator startup route. Once launched, screening, evidence inspection,
report export and live decoding comparison are operated entirely through the browser UI.
A local standalone launch is also provided in `app/README.md` and `start-app.sh`.

Use the newest source ZIP and `trialmatch-selected-adapter.zip`. No dataset download or
training is needed. Keep the Colab GPU runtime active while using the app. A100 or L4 is
recommended; actual device and model versions appear in exported reports.

The selected candidate still has known rare-label failures. Educational screening prototype,
not a clinical enrollment decision. Use the built-in synthetic example for demonstration.


## 1. Upload the newest project source ZIP

In [ ]:
from pathlib import Path
import hashlib
import os
import stat
import zipfile
from google.colab import files

uploaded = files.upload()
expected = 'trialmatch-colab-starter.zip'
if expected not in uploaded:
    raise ValueError(f'Upload the updated {expected}.')
bundle_bytes = uploaded[expected]
BUNDLE_SHA256 = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT = Path('/content/trialmatch-training-' + BUNDLE_SHA256[:12])
archive_path = Path('/content') / expected
archive_path.write_bytes(bundle_bytes)
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if not {'app/streamlit_app.py','src/trialmatch/app_runtime.py','requirements-app.txt'} <= set(archive.namelist()):
        raise ValueError('This is an older bundle. Upload the updated ablation bundle.')
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Archive contains an unsafe path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Archive contains a symbolic link')
    archive.extractall(PROJECT)
del uploaded, bundle_bytes
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/trialmatch-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLCONFIGDIR'] = '/content/trialmatch-matplotlib'
print('Project:', PROJECT)
print('Bundle SHA256:', BUNDLE_SHA256)


## 2. Install the browser app stack and verify CUDA

The cell streams and saves pip output so an install failure includes its real reason.
Use the first upload cell after a runtime reset; PROJECT is defined there. If installation
requests a restart, rerun the upload and setup cells in order.


In [ ]:
import subprocess
import sys
import json

REQUIRE_A100 = False
# Use the uploaded project explicitly; preserve pip's actual error, not just its exit code.
if not (PROJECT / 'requirements-app.txt').is_file() or not (PROJECT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Project setup files are missing. Rerun the updated source ZIP upload cell first.')
INSTALL_LOG = Path('/content/trialmatch-install.log')
command = [sys.executable, '-m', 'pip', 'install',
           '-r', str(PROJECT / 'requirements-app.txt'), '-e', str(PROJECT)]
print('Installing project from:', PROJECT, flush=True)
with INSTALL_LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip installation failed. The real error is above; full log: {INSTALL_LOG}. '
                       'Share that log or its final ERROR lines before changing package versions.')
hardware = json.loads(subprocess.check_output([sys.executable, '-c',
    'import torch,json; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print(json.dumps({"gpu":torch.cuda.get_device_name(0),'
    '"memory_gib":torch.cuda.get_device_properties(0).total_memory/2**30,'
    '"torch":torch.__version__}))'], text=True))
print(hardware)
print('App dependencies installed. Next upload the selected adapter.')
if REQUIRE_A100 and 'A100' not in hardware['gpu']:
    raise RuntimeError('This session is not using A100. Change the runtime GPU, or explicitly set REQUIRE_A100=False.')


## 3. Upload and verify the selected adapter ZIP

In [ ]:
import hashlib
import zipfile
import stat

uploaded = files.upload()
name = 'trialmatch-selected-adapter.zip'
if name not in uploaded:
    raise ValueError('Upload ' + name)
archive_path = Path('/content') / name
archive_path.write_bytes(uploaded[name])
ADAPTER = Path('/content/trialmatch-selected-adapter')
ADAPTER.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        if not (ADAPTER / entry.filename).resolve().is_relative_to(ADAPTER.resolve()):
            raise ValueError('Unsafe archive path')
        if stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError('Symbolic link in archive')
    archive.extractall(ADAPTER)
del uploaded
candidate = json.loads((PROJECT / 'configs/model_candidate.json').read_text())
for name, expected in candidate['adapter_file_sha256'].items():
    if hashlib.sha256((ADAPTER / name).read_bytes()).hexdigest() != expected:
        raise ValueError('Wrong adapter file: ' + name)
print('Selected adapter integrity passed.')


## 4. Start the app server

This starts a background server bound to this runtime. Model weights load only when you
first click screening or comparison in the app; the first use downloads the pinned base
weights and may take time. All model calls use real custom decoders, not simulated outputs.
Inputs/results are session state, not written to a patient database. Download reports from
the app to retain them; closing/resetting the runtime discards the browser session state.


In [ ]:
import os
import subprocess
import sys
import time
import urllib.request

if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    print('This notebook already has an app process; reusing it.')
else:
    environment = os.environ.copy()
    environment['TRIALMATCH_ADAPTER_DIR'] = str(ADAPTER)
    environment['MPLCONFIGDIR'] = '/content/trialmatch-matplotlib'
    APP_LOG_PATH = Path('/content/trialmatch-app.log')
    APP_LOG = APP_LOG_PATH.open('w')
    APP_PROCESS = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', str(PROJECT / 'app/streamlit_app.py'),
         '--server.port', '8501', '--server.address', '0.0.0.0',
         '--server.headless', 'true', '--browser.gatherUsageStats', 'false'],
        cwd=PROJECT, env=environment, stdout=APP_LOG, stderr=subprocess.STDOUT)
    for _ in range(60):
        if APP_PROCESS.poll() is not None:
            raise RuntimeError(APP_LOG_PATH.read_text())
        try:
            with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health', timeout=1) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(.5)
    else:
        raise RuntimeError('Server did not become ready. Inspect /content/trialmatch-app.log')
print('Server is ready. Open the app in the next cell.')


## 5. Open the browser interface

Google Colab's supported iframe proxy is used instead of publishing a public tunnel.
Browser cookie/WebSocket restrictions may affect embedded apps; this integration must be
verified in your account. If the interface cannot connect, download the server log from
cell 6 and use the documented local standalone launch as the fallback. Colab's older
new-window proxy is deprecated, so it is not used here.

In the app: load the synthetic example, choose a decoding mode, screen criteria, inspect
cited sentences and download the HTML report. Use the live comparison button to benchmark
one entered criterion across all three modes. No code editing is needed for those actions.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=1050)


## 6. Optional diagnostics and shutdown

Leave these cells unrun while using the app. Server logs contain technical diagnostics;
entered patient notes are not deliberately logged by the application. After model errors,
retain the error shown in the browser and this log for troubleshooting.


In [ ]:
DOWNLOAD_SERVER_LOG = False
if DOWNLOAD_SERVER_LOG:
    from google.colab import files
    files.download('/content/trialmatch-app.log')


In [ ]:
STOP_APP = False
if STOP_APP and 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    APP_PROCESS.wait(timeout=15)
    APP_LOG.close()
    print('App stopped.')
